# 🏥 Hermes-CDSS 全功能 Colab 演示

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariskang/Tao-CDSS/blob/claude/project-review-optimization-2m39gj/notebooks/hermes_colab_demo.ipynb)


**方言友好 · 语音优先 · 状态感知 · 可审计** 的医疗预问诊 + 医生侧 CDSS 智能体系统。

本 Notebook 一键启动 Gradio 前端(附 ngrok 公网链接),集成:
- 🩺 **患者预问诊**: 多轮自由对话,🎤 语音输入(faster-whisper large-v3, GPU)+ 🔊 语音回复(edge-tts);红旗急症自动升级、心理危机固定话术旁路、近音药名双确认、数字回读
- 👨‍⚕️ **医生工作台**: SOAP/鉴别/证据视图、must-not-miss 裁决、drug_safety 结构化剂量回填、摘要放行(带语音)
- 📜 **伤寒论问答**: 方证匹配(split-conformal 弃权)/鉴别/禁忌/条文检索(BM25+RRF),患者角色自动递归脱敏
- 🛡️ **评测面板**: 红队回归(注入率=0/剂量泄漏=0)、self-play 仿真、SP 回放、金标准校准(bootstrap CI)

> ⚠️ 全部临床规则为工程占位(PENDING_PHYSICIAN_REVIEW),**不构成医疗建议**。
> 建议运行时: **A100 / RTX A6000(96GB)**——whisper large-v3 float16 仅需 ~3GB 显存,大显存可同时挂本地 LLM。


## 1️⃣ 检查 GPU


In [ ]:
!nvidia-smi


## 2️⃣ 获取代码与安装依赖(约 2-3 分钟)


In [ ]:
import os, subprocess
REPO_URL = 'https://github.com/pariskang/Tao-CDSS.git'
# 优先 PR 分支,不存在(已合并删除)则自动回退 main
BRANCHES = ['claude/project-review-optimization-2m39gj', 'main']

if not os.path.exists('Tao-CDSS'):
    for br in BRANCHES:
        r = subprocess.run(['git', 'clone', '--depth', '1', '-b', br, REPO_URL])
        if r.returncode == 0:
            print(f'✅ 已获取分支: {br}')
            break
%cd Tao-CDSS

# 核心 + 演示依赖(与 pyproject [demo] 组一致)
!pip install -q pydantic pyyaml litellm pytest pytest-cov \
    gradio faster-whisper edge-tts pyngrok


## 3️⃣ 快速自检(可选,约 15 秒)
全量 600 项测试;安全三模块(剂量出站/红旗/审计链)要求 100% 覆盖。


In [ ]:
!python -m pytest tests/ -q --tb=no | tail -3


## 4️⃣ 配置(可选)
- **LLM 增强**(不配置则走确定性 fallback,全功能仍可用):任意 litellm 模型
- **ngrok**: 在 https://dashboard.ngrok.com/get-started/your-authtoken 免费获取 token;不填则用 gradio 自带 share 链接


In [ ]:
import os
from getpass import getpass

# ---- LLM(可选) ----
USE_LLM = False  # 改 True 并填 key 以启用 LLM 增强(摘要/抽取/多评审)
if USE_LLM:
    os.environ['HERMES_LLM_MODEL'] = 'anthropic/claude-sonnet-5'
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')
    os.environ['HERMES_LLM_CONSISTENCY_K'] = '3'  # k采样一致性门控

# ---- ngrok(推荐) ----
NGROK_TOKEN = getpass('ngrok authtoken(直接回车跳过): ').strip()
if NGROK_TOKEN:
    os.environ['NGROK_AUTHTOKEN'] = NGROK_TOKEN


## 5️⃣ 预热语音模型(首次约 1 分钟,下载 whisper large-v3)


In [ ]:
from apps.colab_demo import voice
_ = voice._whisper()
print('✅ ASR 就绪(faster-whisper large-v3, GPU float16)')
test_mp3 = voice.synthesize('您好,我是预问诊助理。')
print('✅ TTS 就绪:', test_mp3)


## 6️⃣ 启动前端 🚀
启动后点击输出中的 **ngrok 公网地址**(或 gradio share 链接)即可在手机/电脑浏览器测试。

**体验路径建议**:
1. 患者端选 `emergency_triage` → 开始新会话 → 🎤 说「心口疼得厉害,还出冷汗」→ 观察 E1 升级 + 语音固定话术
2. 新会话说「咳嗽两天」→ 逐问回答到 DOCTOR_REVIEW → 医生工作台生成视图 → adopt 放行,听患者摘要语音
3. 伤寒问答: 「桂枝汤和麻黄汤怎么鉴别?」;切 patient 角色看递归脱敏
4. 评测面板跑 `redteam` 验证注入率=0、剂量泄漏=0


In [ ]:
from apps.colab_demo.app import launch
launch()  # 阻塞运行;停止请中断本单元格


## 附: 故障排查
- **麦克风无声**: 浏览器需允许麦克风权限;ngrok 免费版首次访问需点 Visit Site
- **TTS 无声**: edge-tts 需外网;Colab 默认可用,内网环境自动降级为纯文本
- **显存不足**: `voice.py` 会自动回退 CPU int8(small 模型),或手动改 `WhisperModel('medium', ...)`
- **LLM 400/超时**: 网关自动指数退避重试并降级确定性 fallback,问诊安全路径(红旗/剂量/审计)零 LLM 依赖,不受影响
